In [30]:
import os
import json
import asyncio
from typing import List
from agents import function_tool, Agent, Runner
from exa_py import Exa
import dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput

In [3]:
dotenv.load_dotenv()
EXA_API_KEY = os.environ.get("EXA_API_KEY")
if not EXA_API_KEY:
    raise RuntimeError("Missing EXA_API_KEY. Set it in your environment.")
exa = Exa(api_key=EXA_API_KEY)

In [4]:
@function_tool
def search_code_context(
    query: str,
    num_results: int = 3,
    max_chars_per_doc: int = 1200,
    summarize: bool = True
) -> str:
    """
    Searches code-related sources (GitHub, docs, Q&A) and returns compact context
    for coding questions.

    Args:
        query: Natural language query (e.g., "Express.js middleware for authentication").
        num_results: How many results to fetch (default 3).
        max_chars_per_doc: Max characters of content per result (default 1200).
        summarize: Ask Exa to include a short summary for each result.

    Returns:
        A JSON string with a list of {title, url, published_date, summary, text}.
        The agent will read and use this to craft an answer.
    """
    # Grab both metadata and a chunk of page contents so the LLM has real context
    resp = exa.search_and_contents(
        query=query,
        num_results=num_results,
        # Optional knobs: include_domains=[...], start_published_date="2024-01-01"
        contents={
            "max_characters": max_chars_per_doc,
            "include_html": False,
            "summary": summarize,
        },
    )

    out: List[dict] = []
    for r in resp.results:
        out.append({
            "title": getattr(r, "title", None),
            "url": getattr(r, "url", None),
            "published_date": getattr(r, "published_date", None),
            "summary": getattr(r, "summary", None) if hasattr(r, "summary") else None,
            "text": getattr(r, "text", None),
        })

    return json.dumps(out, ensure_ascii=False)

In [31]:
groq_api_key = os.getenv('GROQ_API_KEY')
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
gpt_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [32]:
agent = Agent(
    model=gpt_model,
    name="Coding Assistant",
    instructions=(
        "You are an expert coding assistant. "
        "Whenever a query asks for examples, libraries, APIs, or implementation details, "
        "first call the search_code_context tool to gather relevant snippets and docs. "
        "Then synthesize a clear, working answer with code."
    ),
    tools=[search_code_context],
    
)

In [33]:
user_task = "How to use the requests library to make a POST request in Python?"
result = await Runner.run(agent, input=user_task, max_turns=3)
print("\n=== Final Answer ===\n")
print(result.final_output)

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: gsk_KvXu********************************************DG4F. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}



=== Final Answer ===

Below is a concise, ready‑to‑run guide for sending POST requests with **`requests`** in Python. It covers the most common scenarios you’ll run into:

| What you want to do | Code snippet | Explanation |
|---------------------|--------------|-------------|
| **Simple form‑encoded POST** (key‑value pairs) | ```python\nimport requests\n\nurl = "https://httpbin.org/post"          # any endpoint that accepts POST\npayload = {"username": "alice", "age": 30}\n\nresp = requests.post(url, data=payload)   # `data` → form‑encoded body\nprint(resp.status_code)\nprint(resp.json())\n``` | `data=` sends the dict as *application/x‑www‑form‑urlencoded* (the same format a normal HTML form uses). |
| **POST JSON** (most modern APIs) | ```python\nimport requests\n\nurl = "https://httpbin.org/post"\njson_body = {"title": "Hello", "price": 9.99}\n\nresp = requests.post(url, json=json_body)   # `json=` handles encoding & sets header\nprint(resp.status_code)\nprint(resp.json())\n``` | `

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: gsk_KvXu********************************************DG4F. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}
